In [1]:
# Run surrogate predictions

from gpPredict import *
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import os

import pickle


# Load pickle file with logistic regression
with open('log_reg_model.pkl', 'rb') as f:
    failure_mode_selection = pickle.load(f)
    # First parameter is the aspect ratio
    # Second parameter is Vp/Vs

    # Codes:
    # 0 = Flexure
    # 1 = Shear

# Test the failure mode selection model
failure_mode_selection.predict(np.array([[1.0, 0.6]]))

array([1], dtype=int8)

In [ ]:
# ndParams = ['ar', 'lrr', 'srr', 'alr', 'sdr', 'smr']
# ndParams = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

def get_BW_params(ndParams):

    # Define the failure mode using the logistic regression model
    failure_mode = failure_mode_selection.predict(np.array([[ndParams[0], ndParams[5]]]))[0]

    # Define the surrogate_file and input_json based on the failure mode
    if failure_mode == 0:
        surrogate_file = os.path.join('gpModelFlexure', 'SimGpModel.json')
        input_json = os.path.join('gpModelFlexure', 'scInput.json')
    else:
        surrogate_file = os.path.join('gpModelShear', 'SimGpModel.json')
        input_json = os.path.join('gpModelShear', 'scInput.json')

    params_list = [
        ["RV_column1", ndParams[0]],
        ["RV_column2", ndParams[1]],
        ["RV_column3", ndParams[2]],
        ["RV_column4", ndParams[3]],
        ["RV_column5", ndParams[4]],
        ["RV_column6", ndParams[5]]
    ]

    output = main(params_list, [], surrogate_file, 'dummyout.out', input_json)
    
    # bw model parameters are the all the output parameters except the last one
    bw_model_params = output[0][:-1]

    # min error is the last output parameter
    min_error = output[0][-1]
    
    return bw_model_params, min_error


0.0504900779777598